# Reproducing Nishimoto et al. (2011) from CRCNS vim-2

**Paper.** Nishimoto, Vu, Naselaris, Benjamini, Yu & Gallant (2011), "Reconstructing Visual Experiences from Brain Activity Evoked by Natural Movies", Current Biology 21, 1641-1646, doi:10.1016/j.cub.2011.08.031.

**Claims file.** `claims/nishimoto-2011-reconstructing.yaml`, written from the version of record (main text, legends, Experimental Procedures) and its Supplemental Information.

**Dataset.** CRCNS vim-2 (Gallant Lab Natural Movie 4T fMRI Data), doi:10.6080/K00Z715X, read from the public S3 bucket `crcnsarchive` under `vim-2/`. CRCNS data sets are not versioned, so the version is recorded as `unversioned` (docs/archives.md); there is only one published version to pin, and it is the one the data descriptor (version 0.61, 2015-02-17) describes as holding the data of this paper. The MD5 sums of the files read, from the data set's `checksums.md5`, are printed in the data check below so that a later change can be detected: Stimuli.tar.gz d06069177dd2ccad303009b7003c67b9, VoxelResponses_subject1.tar.gz 0e389bfa0b2235d1763d8a0d3bebf0af, VoxelResponses_subject2.tar.gz 92f3e621677ac1c8c226c87d77977c88, VoxelResponses_subject3.tar.gz 3ea1084138f70583228ad5649c935921. The raw per-session data (`raw_data/`) are not used: the paper's analysis starts from the preprocessed responses these files hold.

Each file is a tar.gz holding one MATLAB 7.3 (HDF5) file. It is streamed once from the bucket into memory and opened there (vim2.py); nothing is written to disk.

| Claim | Tested here |
|---|---|
| fig2de-prediction-accuracy | tested |
| fig2e-directional-gain-small | tested |
| figs1a-motion-better-every-area | tested |
| fig2fgh-retinotopic-receptive-fields | blocked, too_large (time) |
| fig2j-speed-eccentricity | blocked, too_large (time) |
| fig3ab-identification-s1 | tested |
| fig3c-identification-large-set | blocked, data_missing (1,000,000-clip set not public) |
| fig4-reconstructions | blocked, data_missing (sampled prior not public) |
| fig4d-reconstruction-accuracy | blocked, data_missing (sampled prior not public) |
| discussion-color-no-gain | blocked, too_large (time) |

**Main deviation.** The paper fits each voxel with L1-regularized least squares by coordinate descent. With 26,220 regressors, about 4,100 voxels per subject and three models, that did not fit the time of this run, so every model here is fit by ridge regression, solved through the kernel of the training features, with the penalty chosen per voxel by prediction of a held-out 10% of the training data in 50 s blocks (the claims file's reading a). Reading b (several cross-validation splits) was not run, for time.


In [ ]:
import time, resource, collections, urllib.request
import numpy as np
from scipy import stats
import vim2

T0 = time.time()
SEED = 2011  # standard choice; each claim's random draws use their own generator
ALPHAS = np.logspace(1, 7, 13).astype(np.float32)  # ridge penalties searched per voxel
MODELS = ["static", "nondirectional", "directional"]
SUBJECTS = [1, 2, 3]
def mem_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6

## Stimuli and motion-energy features

Frames are downsampled to 96 x 96, converted to CIE L*A*B*, and the luminance passes through the 6,555-filter motion-energy bank (Supplemental, Motion-Energy Encoding Model; Gabor Wavelet Basis Set). The bank is pymoten's pyramid; its default envelopes give the paper's count of 6,555 filters (see vim2.py).

In [ ]:
t = time.time()
S = vim2.load_stimuli(("L",))
stim_info = {k: S[k] for k in ("member", "member_bytes", "layout", "st_range", "sv_range")}
L_range = {k: (float(S[k]["L"].min()), float(S[k]["L"].max())) for k in ("st", "sv")}
t_load = time.time() - t
pyr = vim2.pyramid()
filters = pyr.filters
t = time.time()
me_test = vim2.motion_energy(S["sv"]["L"], filters, block=900)
me_train = vim2.motion_energy(S["st"]["L"], filters, block=9000)
t_me = time.time() - t
del S
print(f"stimuli streamed and converted in {t_load:.0f} s; motion energy in {t_me:.0f} s; peak memory {mem_gb():.1f} GB")

In [ ]:
md5 = urllib.request.urlopen(vim2.BASE + "checksums.md5", timeout=60).read().decode()
print("MD5 sums of the files read (checksums.md5):")
for line in md5.splitlines():
    if "Stimuli" in line or "VoxelResponses" in line:
        print(" ", line)
print("Stimulus member:", stim_info["member"], stim_info["member_bytes"], "bytes")
print("Stored arrays:", stim_info["layout"])
print("Training frames", stim_info["layout"]["st"][0][0], "= 7200 s x 15 Hz:", stim_info["layout"]["st"][0][0] == 7200 * 15)
print("Test frames", stim_info["layout"]["sv"][0][0], "= 540 s x 15 Hz:", stim_info["layout"]["sv"][0][0] == 540 * 15)
print("Pixel range (uint8), training", stim_info["st_range"], "test", stim_info["sv_range"])
print("Luminance L* range after 96 x 96 downsampling, training %.1f to %.1f, test %.1f to %.1f (0 to 100 expected)" % (L_range["st"] + L_range["sv"]))
print("Filters in the bank:", len(filters), "(paper: 6,555)")
c = collections.Counter((f["spatial_freq"], f["temporal_freq"]) for f in filters)
print("Filters per (spatial frequency c/image, temporal frequency Hz):", {k: v for k, v in sorted(c.items())})
print("Motion energy at 1 Hz, training", me_train.shape, "test", me_test.shape,
      "range %.2f to %.2f (log units)" % (me_train.min(), me_train.max()))
ktr = vim2.keep_mask(7200, vim2.TRAIN_BLOCK)
kte = vim2.keep_mask(540, vim2.TEST_BLOCK)
print("Samples kept after discarding the first 6 s of each block: training", ktr.sum(), "(paper 7,128), test", kte.sum(), "(paper 486)")

In [ ]:
feats = {}
for m in MODELS:
    cols = vim2.model_columns(filters, m)
    Xtr, Xte = vim2.build_features(me_train, me_test, cols)
    feats[m] = (vim2.delayed(Xtr, vim2.DELAYS, 600)[ktr], vim2.delayed(Xte, vim2.DELAYS, 60)[kte])
    print(f"{m}: {len(cols)} features x {len(vim2.DELAYS)} delays = {feats[m][0].shape[1]} regressors")
# single-delay directional features, for the clock check and the 4 s model of identification
cols = vim2.model_columns(filters, "directional")
Xtr1, Xte1 = vim2.build_features(me_train, me_test, cols)
single = {d: (vim2.delayed(Xtr1, (d,), 600)[ktr], vim2.delayed(Xte1, (d,), 60)[kte]) for d in (0, 4, 8)}
del Xtr1, Xte1

## Responses, fits and the data check, per subject

For each subject the V1, V2, V3, V3A and V3B voxels are taken from the data set's ROI masks. Voxels with any missing value are dropped and counted. The test responses are the ten repeats (`rva`); following Data Preprocessing ("signals were then normalized to mean 0.0 and standard deviation 1.0" for each run), each repeat is z-scored per voxel, and the observed test signal of reading (a) is their mean. Each model is fit on 90% of the training samples, the penalty of each voxel is the one that best predicts the held-out 10% (50 s blocks, Voxel Selection), and the model is refit on all training samples with that penalty.

The clock check fits the directional model at a single delay of 0, 4 or 8 s: if responses and stimuli share a clock, the 4 s delay (the hemodynamic lag the paper assumes) predicts best and 0 s worst.

In [ ]:
def fit_subject(subj):
    t = time.time()
    d = vim2.load_subject(subj)
    t_load = time.time() - t
    good = ~(np.isnan(d["rt"]).any(0) | np.isnan(d["rva"]).any((0, 1)) | np.isnan(d["rv"]).any(0))
    Y = d["rt"][ktr][:, good]
    rva = d["rva"][:, :, good]
    rv_check = float(np.nanmax(np.abs(d["rv"][:, good] - d["rva"][:, :, good].mean(0))))
    raw_std = (float(d["rt"][:, good].std()), float(d["rv"][:, good].std()), float(rva.std()))
    first6 = float(d["rt"][~ktr][:, good].mean())
    rva = (rva - rva.mean(1, keepdims=True)) / rva.std(1, keepdims=True)
    Yrep = rva[:, kte]
    Yte = Yrep.mean(0)
    held = vim2.holdout_blocks(Y.shape[0], np.random.default_rng([SEED, subj]))
    out = dict(subject=subj, t_load=t_load, n_roi=len(d["vox"]), n_drop=int((~good).sum()),
               area=d["area"][good], hemi=d["hemi"][good], roi_counts=d["roi_counts"],
               n_any_roi=d["n_voxels_any_roi"], ei=d["ei"], layout=d["layout"],
               roi_values=d["roi_values"], n_multi=d["n_multi_label"], rv_check=rv_check,
               raw_std=raw_std, first6=first6, rest_mean=float(Y.mean()),
               n_held=int(held.sum()), acc={}, alpha={})
    for m in MODELS:
        Xtr, Xte = feats[m]
        P = vim2.kernel_ridge_path(Xtr[~held], Y[~held], Xtr[held], ALPHAS)
        rh = np.array([vim2.corr_cols(P[i], Y[held]) for i in range(len(ALPHAS))])
        ai = rh.argmax(0)
        out["alpha"][m] = np.bincount(ai, minlength=len(ALPHAS))
        if m == "directional":
            out["held_acc"] = rh.max(0)
            out["Pheld"] = P[ai, :, np.arange(P.shape[2])].T
            out["ai"] = ai
            Pte, R = vim2.ridge_fit_predict(Xtr, Y, Xte, ai, ALPHAS, residuals=True)
            out["R"] = R
            out["Pte"] = Pte
        else:
            Pte = vim2.ridge_fit_predict(Xtr, Y, Xte, ai, ALPHAS)
        out["acc"][m] = {"avg": vim2.corr_cols(Pte, Yte),
                         "rep": np.mean([vim2.corr_cols(Pte, Yrep[k]) for k in range(10)], 0)}
    out["delay_acc"] = {}
    for dl, (X1, X1e) in single.items():
        P = vim2.kernel_ridge_path(X1, Y, X1e, ALPHAS[[6, 8]])
        out["delay_acc"][dl] = max(float(vim2.corr_cols(P[i], Yte).mean()) for i in range(2))
    if subj == 1:
        out["Y"], out["Yte"], out["Yrep"], out["held"] = Y, Yte, Yrep, held
    out["t_total"] = time.time() - t
    return out

RES = {}
for s in SUBJECTS:
    RES[s] = fit_subject(s)
    r = RES[s]
    print(f"subject {s}: loaded in {r['t_load']:.0f} s, fit in {r['t_total']:.0f} s, peak memory {mem_gb():.1f} GB")

In [ ]:
for s in SUBJECTS:
    r = RES[s]
    print(f"--- subject {s}")
    print("  stored arrays:", r["layout"], "ei:", r["ei"])
    print("  ROI mask values:", r["roi_values"], "; voxels in any ROI:", r["n_any_roi"], "of 73,728")
    print("  V1-V3B voxels per mask:", {k: v for k, v in r["roi_counts"].items() if k[:2] in ("v1", "v2", "v3")})
    print("  voxels in more than one V1-V3B mask:", r["n_multi"])
    print(f"  V1-V3B voxels {r['n_roi']} (paper about 4,500 per subject); dropped for missing values {r['n_drop']}; kept {r['n_roi'] - r['n_drop']}")
    print("  kept per area:", dict(collections.Counter(r["area"])))
    print(f"  rv equals the mean of the ten rva repeats: max abs difference {r['rv_check']:.2e}")
    print(f"  s.d. of rt {r['raw_std'][0]:.3f}, rv {r['raw_std'][1]:.3f}, rva {r['raw_std'][2]:.3f} (training z-scored per run; test in other units, so each repeat is z-scored here)")
    print(f"  mean training response in the first 6 s of each run {r['first6']:.3f}, in the kept samples {r['rest_mean']:.3f} (onset transient, discarded as in the paper)")
    print(f"  held-out training samples {r['n_held']} of {ktr.sum()}")
    for m in MODELS:
        print(f"  {m}: voxels per chosen penalty {dict(zip(np.round(ALPHAS).astype(int).tolist(), r['alpha'][m].tolist()))}")
    print("  clock check, mean test accuracy of the directional model at one delay:",
          {f"{k} s": round(v, 3) for k, v in r["delay_acc"].items()})

## fig2de-prediction-accuracy

> Averaged over subjects and voxels in V1, V2, V3, V3A and V3B, the prediction accuracy on the test data is 0.24 for the static model, 0.39 for the nondirectional motion-energy model and 0.40 for the directional motion-energy model, and the differences are significant (Wilcoxon signed-rank test, p < 0.0001).

Readings: accuracy on (a) the repeat-averaged test signal or (b) the mean over repeats of the single-repeat correlation; subjects combined by (i) pooling voxels or (ii) the mean of the three subject means. Tolerances 0.024, 0.039, 0.040.

In [ ]:
EXP = {"static": (0.24, 0.024), "nondirectional": (0.39, 0.039), "directional": (0.40, 0.040)}
def pooled(m, rd):
    return np.concatenate([RES[s]["acc"][m][rd] for s in SUBJECTS])
fig2de = {}
for rd in ("avg", "rep"):
    p_ds = stats.wilcoxon(pooled("directional", rd), pooled("static", rd)).pvalue
    p_dn = stats.wilcoxon(pooled("directional", rd), pooled("nondirectional", rd)).pvalue
    for comb in ("pooled", "subject means"):
        if comb == "pooled":
            means = {m: float(pooled(m, rd).mean()) for m in MODELS}
        else:
            means = {m: float(np.mean([RES[s]["acc"][m][rd].mean() for s in SUBJECTS])) for m in MODELS}
        within = all(abs(means[m] - EXP[m][0]) <= EXP[m][1] for m in MODELS)
        order = means["static"] < means["nondirectional"] < means["directional"]
        fig2de[(rd, comb)] = dict(means=means, within=within, order=order, p_ds=p_ds, p_dn=p_dn)
        print(f"reading ({'a' if rd == 'avg' else 'b'}) {rd}, ({'i' if comb == 'pooled' else 'ii'}) {comb}: "
              + ", ".join(f"{m} {means[m]:.3f}" for m in MODELS)
              + f"; within tolerance {within}; order {order}; Wilcoxon dir vs static p = {p_ds:.1e}, dir vs nondir p = {p_dn:.1e}")
print()
print("Per subject, reading (a):")
for s in SUBJECTS:
    a = RES[s]["acc"]
    ps = stats.wilcoxon(a["directional"]["avg"], a["static"]["avg"]).pvalue
    pn = stats.wilcoxon(a["directional"]["avg"], a["nondirectional"]["avg"]).pvalue
    print(f"  S{s} (n = {len(a['static']['avg'])} voxels): " + ", ".join(f"{m} {a[m]['avg'].mean():.3f}" for m in MODELS)
          + f"; p dir-static {ps:.1e}, dir-nondir {pn:.1e}")
print("Null reference: a voxel's correlation on 486 samples exceeds 0.12 by chance with p < 0.01 (two-sided, normal approximation):",
      round(float(stats.norm.ppf(0.995) / np.sqrt(486 - 3)), 3))
print("Range of voxel accuracies (reading a, pooled):", {m: (round(float(pooled(m, 'avg').min()), 3), round(float(pooled(m, 'avg').max()), 3)) for m in MODELS})

**Result: reproduced.** Under the repeat-averaged test signal (reading a), the pooled means are 0.217 (static), 0.375 (nondirectional) and 0.379 (directional), each within its tolerance of 0.24, 0.39 and 0.40, in the paper's order, and both paired Wilcoxon tests give p below 0.0001 (printed as 0). The mean of subject means gives the same three values to three decimals. Under reading b (single repeats) the order and tests hold but the means are about half (0.120, 0.207, 0.209), outside tolerance. The static mean, 0.217, is 0.023 below 0.24, close to the edge of its tolerance of 0.024. The fits use ridge regression in place of the paper's L1 regression.

## fig2e-directional-gain-small

> The directional model predicts better than the nondirectional model, but the difference is small, which suggests that BOLD signals carry spatially localized but mostly nondirectional motion information.

Reproduced when the directional mean exceeds the nondirectional mean, the paired Wilcoxon test gives p < 0.0001, and the difference is less than a tenth of the gain of the nondirectional model over the static model.

In [ ]:
for (rd, comb), v in fig2de.items():
    mm = v["means"]
    gain_dir = mm["directional"] - mm["nondirectional"]
    gain_motion = mm["nondirectional"] - mm["static"]
    ok = gain_dir > 0 and v["p_dn"] < 1e-4 and gain_dir < 0.1 * gain_motion
    print(f"reading {rd}, {comb}: directional - nondirectional = {gain_dir:.4f}; nondirectional - static = {gain_motion:.4f}; "
          f"ratio {gain_dir / gain_motion:.3f}; Wilcoxon p = {v['p_dn']:.1e}; criterion met {ok}")
for s in SUBJECTS:
    a = RES[s]["acc"]
    dd = a["directional"]["avg"] - a["nondirectional"]["avg"]
    print(f"  S{s}: mean difference {dd.mean():.4f}, fraction of voxels with directional better {np.mean(dd > 0):.2f}")

**Result: reproduced.** Directional minus nondirectional accuracy is 0.0042 against a gain of 0.158 of the nondirectional over the static model (a ratio of 0.026, under the bound of 0.1), with Wilcoxon p below 0.0001, under every reading; the directional model is better in 56 to 62% of voxels per subject.

## figs1a-motion-better-every-area

> In each of the five visual areas and each of the three subjects, both motion models predict better than the static model, and the best predictions are in V1 for all three subjects (Wilcoxon rank-sum test, p < 0.0001).

Mean accuracy per model, area and subject (reading a of the test signal, with reading b shown too). Rank-sum readings: (a) V1 against each other area in turn; (b) V1 against the other four pooled. Model readings: (i) directional; (ii) each of the three models.

In [ ]:
AREAS = vim2.AREAS
for rd in ("avg", "rep"):
    print(f"=== test-signal reading {rd}")
    n_cells = 0
    n_beat = 0
    v1_best = {}
    for s in SUBJECTS:
        r = RES[s]
        for ar in AREAS:
            sel = r["area"] == ar
            mm = {m: r["acc"][m][rd][sel].mean() for m in MODELS}
            beat = mm["nondirectional"] > mm["static"] and mm["directional"] > mm["static"]
            n_cells += 1
            n_beat += beat
            print(f"  S{s} {ar:4s} n={sel.sum():4d}: " + ", ".join(f"{m} {mm[m]:.3f}" for m in MODELS) + f"; both motion > static {beat}")
        for m in MODELS:
            means = {ar: r["acc"][m][rd][r["area"] == ar].mean() for ar in AREAS}
            best = max(means, key=means.get)
            v1 = r["acc"][m][rd][r["area"] == "v1"]
            p_each = {ar: stats.ranksums(v1, r["acc"][m][rd][r["area"] == ar], alternative="greater").pvalue for ar in AREAS[1:]}
            p_pool = stats.ranksums(v1, r["acc"][m][rd][r["area"] != "v1"], alternative="greater").pvalue
            v1_best[(s, m)] = (best == "v1", max(p_each.values()) < 1e-4, p_pool < 1e-4)
            print(f"    S{s} {m}: best area {best}; rank-sum V1 > each other area, largest p = {max(p_each.values()):.1e} "
                  f"({ {k: f'{v:.0e}' for k, v in p_each.items()} }); V1 > other four pooled p = {p_pool:.1e}")
    print(f"  cells where both motion models beat static: {n_beat} of {n_cells}")
    for m in MODELS:
        for k, lab in ((1, "(a) each area"), (2, "(b) pooled")):
            n_ok = sum(v1_best[(s, m)][0] and v1_best[(s, m)][k] for s in SUBJECTS)
            print(f"  {m}, rank-sum reading {lab}: subjects with V1 best and p < 0.0001: {n_ok} of 3")

**Result: reproduced.** Both motion models beat the static model in all 15 area and subject cells, under both test-signal readings. V1 has the highest mean in every subject for every model. With V1 compared with the other four areas pooled (rank-sum reading b), p is below 0.0001 in all three subjects for every model (largest 1.2e-11, S3 directional). With V1 compared with each other area in turn (reading a), this holds in all three subjects for the static model only: for the directional model V1 against V3A gives p = 1.3e-04 in S1 and 3.2e-02 in S3.

## fig3ab-identification-s1

> For subject S1 the motion-energy model identifies the test second that evoked an observed BOLD pattern within one volume (1 s) 95% of the time (464 of 486 volumes), far above chance (under 1%, 3 of 486).

The 2,000 V1-V3B voxels whose model fit on 90% of the training data best predicts the held-out 10% are used (Voxel Selection). The likelihood is the multivariate Gaussian of Likelihood Estimation, with the covariance of the training residuals; the Tikhonov penalty is chosen from a grid by identification of the held-out 10% of the training data (claims file, Common methods), using the residuals of the 90% fit. Readings of the model: (a) four delays, (b) a single 4 s delay refit. Readings of the observed signal: (i) the mean of the ten repeats, (ii) each repeat identified on its own, the count averaged over repeats. Expected 464, tolerance 46.4 (at least 418).

In [ ]:
r1 = RES[1]
Y, Yte, Yrep, held = r1["Y"], r1["Yte"], r1["Yrep"], r1["held"]
tt = np.arange(540)[kte]
th = np.arange(len(held))[held]
LAMS = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]
def within1(choice, times):
    return int(np.sum(np.abs(times[choice] - times) <= 1))
chance = float(np.mean([np.sum(np.abs(tt - t0) <= 1) for t0 in tt]))
print(f"chance count within one volume: {chance:.2f} of 486 ({100 * chance / 486:.2f}%)")
ident = {}
for mdl in ("a", "b"):
    if mdl == "a":
        Xtr, Xte = feats["directional"]
        held_acc, Pheld_all, ai = r1["held_acc"], r1["Pheld"], r1["ai"]
    else:
        Xtr, Xte = single[4]
        P = vim2.kernel_ridge_path(Xtr[~held], Y[~held], Xtr[held], ALPHAS)
        rh = np.array([vim2.corr_cols(P[i], Y[held]) for i in range(len(ALPHAS))])
        ai = rh.argmax(0)
        held_acc = rh.max(0)
        del P
    sel = np.argsort(held_acc)[::-1][:2000]
    Pheld, R90 = vim2.ridge_fit_predict(Xtr[~held], Y[~held], Xtr[held], ai, ALPHAS, residuals=True)
    cov90 = (R90[:, sel].T @ R90[:, sel]) / R90.shape[0]
    hc = [within1(vim2.identify(Y[held][:, sel], Pheld[:, sel], cov90, l), th) for l in LAMS]
    lam = LAMS[int(np.argmax(hc))]
    Pte, R = vim2.ridge_fit_predict(Xtr, Y, Xte, ai, ALPHAS, residuals=True)
    cov = (R[:, sel].T @ R[:, sel]) / R.shape[0]
    res_lam = {}
    for l in LAMS:
        c_avg = within1(vim2.identify(Yte[:, sel], Pte[:, sel], cov, l), tt)
        c_rep = [within1(vim2.identify(Yrep[k][:, sel], Pte[:, sel], cov, l), tt) for k in range(10)]
        res_lam[l] = (c_avg, float(np.mean(c_rep)), min(c_rep), max(c_rep))
    ident[mdl] = (lam, res_lam)
    print(f"model reading ({mdl}): held-out accuracy of the 2,000th voxel {held_acc[sel[-1]]:.3f}; "
          f"held-out identification within 1 s by Tikhonov penalty {dict(zip(LAMS, hc))} of {held.sum()}; chosen {lam}")
    for l, (ca, cr, lo, hi) in res_lam.items():
        mark = " <- chosen" if l == lam else ""
        print(f"   penalty {l}: (i) repeat mean {ca} of 486 ({100 * ca / 486:.1f}%); (ii) single repeats mean {cr:.1f} (range {lo} to {hi}){mark}")

**Result: reproduced.** With the four-delay directional model and the repeat-averaged test signal, 424 of 486 test volumes (87.2%) are identified within one volume in S1 at the Tikhonov penalty chosen on the held-out training data, within the tolerance of 464 (at least 418); chance is 2.96 of 486 (0.61%). The count depends on the penalty: 419 at 1.0 and 409 at 10, and 311 at 100. Single repeats give 357 on average (range 333 to 391), and the single 4 s delay model gives 344 (repeat mean) and 266 (single repeats), all below 418 but far above chance.

## Blocked claims

- **fig2fgh-retinotopic-receptive-fields** and **fig2j-speed-eccentricity**: the simulated white-noise and grating probes through the fitted models were not run before the run's time limit (blocked, too_large for time; nothing of them was computed).
- **discussion-color-no-gain**: needs the filter bank on the A* and B* channels as well, three times the feature computation and a fourth model fit per subject; not run in the time (too_large).
- **fig3c-identification-large-set**, **fig4-reconstructions**, **fig4d-reconstruction-accuracy**: need the 1,000,000-clip identification set or the 18,000,000 s sampled prior of YouTube clips. The data set holds neither (the descriptor lists only the experiment's stimuli), and the paper gives no location for them (data_missing).

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {mem_gb():.1f} GB")

## Summary

| Claim | Outcome |
|---|---|
| fig2de-prediction-accuracy | reproduced (0.217, 0.375, 0.379 against 0.24, 0.39, 0.40) |
| fig2e-directional-gain-small | reproduced (gain 0.004 against 0.158) |
| figs1a-motion-better-every-area | reproduced (15 of 15 cells; V1 best in 3 of 3 subjects) |
| fig2fgh-retinotopic-receptive-fields | blocked, too_large (time) |
| fig2j-speed-eccentricity | blocked, too_large (time) |
| fig3ab-identification-s1 | reproduced (424 of 486 against 464) |
| fig3c-identification-large-set | blocked, data_missing |
| fig4-reconstructions | blocked, data_missing |
| fig4d-reconstruction-accuracy | blocked, data_missing |
| discussion-color-no-gain | blocked, too_large (time) |

**How the analysis differs from the paper's.** Ridge regression replaces the L1-regularized regression, with the penalty chosen per voxel on a held-out 10% of the training data; the cross-validated reading of the penalty was not run. The test repeats are z-scored per voxel before averaging, since the data set's test responses are not in the units of the training responses. The filter bank is pymoten's, whose default envelopes give the paper's 6,555 filters.

**What was found about the dataset.** The files match the descriptor in shape (7,200 and 540 samples, 73,728 voxels, 108,000 and 8,100 frames) and `rv` is the mean of `rva`. The descriptor does not say that the responses include the first 6 s of each block that the paper discards (they do: the mean training response there is 0.28 to 0.42, against 0.01 elsewhere), that the test responses are on a different scale from the z-scored training responses (s.d. 0.10 to 0.11 for `rv`, 0.20 to 0.23 for `rva`, against 0.99 for `rt`), or that 29 to 179 V1-V3B voxels per subject hold missing values. The "peak BOLD responses" it mentions are not delay-shifted: the clock check gives 0.31 to 0.35 at a 4 s delay against 0.08 to 0.10 at 0 s. The descriptor also lists only the V1-V4 masks by example; the files hold 30 ROI masks, and one subject 3 voxel lies in two V1-V3B masks.